In [1]:
import sys
print(sys.executable)

import subprocess
import os

BASE_DIR = os.getcwd()
INPUT_DIRECTORY = os.path.join(BASE_DIR, "images", "original")
OUTPUT_BASE_DIRECTORY = os.path.join(BASE_DIR,  "images", "processed")
OUTPUT_MPI_DIRECTORY = os.path.join(OUTPUT_BASE_DIRECTORY, "mpi")
OUTPUT_OMP_DIRECTORY = os.path.join(OUTPUT_BASE_DIRECTORY, "omp")
OUTPUT_HYB_DIRECTORY = os.path.join(OUTPUT_BASE_DIRECTORY, "hyb")
OUTPUT_CUDA_DIRECTORY = os.path.join(OUTPUT_BASE_DIRECTORY, "cuda")
SET_ENV_PATH = os.path.join(BASE_DIR, "..", "set_env.sh")

print(INPUT_DIRECTORY)
print(os.listdir(INPUT_DIRECTORY))

print(SET_ENV_PATH)

import re

from PIL import Image

/usr/local/Anaconda3-2025.06/bin/python
/users/eleves-a/2024/toan.lopez/parallel-gif-filter/images/original
['051009.vince.gif', '1.gif', '200_s.gif', '9815573.gif', 'Campusplan-Hausnr.gif', 'Campusplan-Mobilitaetsbeschraenkte.gif', 'Mandelbrot-large.gif', 'Produits_sous_linux.gif', 'TimelyHugeGnu.gif', 'australian-flag-large.gif', 'fire.gif', 'frame_002.gif', 'giphy-3.gif', 'porsche.gif', 'tumblr_myxlbtwVEb1qzt4vjo1_r14_500_large.gif', 'walla.gif', 'wallace.gif', 'walle.gif']
/users/eleves-a/2024/toan.lopez/parallel-gif-filter/../set_env.sh


In [2]:
def run_command(cmd):
    sourced_cmd = f"source {SET_ENV_PATH} && {cmd}"
    result = subprocess.run(
        sourced_cmd, 
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        shell=True, 
        executable="/bin/bash"
    )
    return result.stdout.decode()

# MPI
def run_sobelf_mpi(input_file, output_file, ranks=16, nodes=1, input_dir=INPUT_DIRECTORY):
    input_path = os.path.join(input_dir, input_file)
    output_path = os.path.join(OUTPUT_MPI_DIRECTORY, output_file)
    
    cmd = f"salloc -N {nodes} -n {ranks} mpirun ./sobelf_mpi {input_path} {output_path}"
    return run_command(cmd)

# OpenMP
def run_sobelf_omp(input_file, output_file, threads=16, input_dir=INPUT_DIRECTORY):
    input_path = os.path.join(input_dir, input_file)
    output_path = os.path.join(OUTPUT_OMP_DIRECTORY, output_file)

    cmd = f"export OMP_NUM_THREADS={threads} && salloc -n 1 ./sobelf_omp {input_path} {output_path}"
    return run_command(cmd)

# OpenMP + MPI
def run_sobelf_hyb(input_file, output_file, ranks=4, threads=4, nodes=1, input_dir=INPUT_DIRECTORY):
    input_path = os.path.join(input_dir, input_file)
    output_path = os.path.join(OUTPUT_HYB_DIRECTORY, output_file)

    cmd = f"export OMP_NUM_THREADS={threads} && salloc -N {nodes} -n {ranks} mpirun ./sobelf_hyb {input_path} {output_path}"
    return run_command(cmd)

# CUDA
def run_sobelf_cuda(input_file, output_file, input_dir=INPUT_DIRECTORY):
    input_path = os.path.join(input_dir, input_file)
    output_path = os.path.join(OUTPUT_CUDA_DIRECTORY, output_file)
    cmd = f"salloc -N 1 -n 1 ./sobelf_cuda {input_path} {output_path}"
    return run_command(cmd)



In [3]:
# Parse output
def parse_sobel_filter_output(output):
    match = re.search(r"SOBEL done in ([0-9.]+) s", output)
    if match:
        return float(match.group(1))
    else:
        raise ValueError("couldn't parse output :\n" + output + '\n')

def profile_mpi(input_file, output_file, ranks=16, nodes=1):
    return parse_sobel_filter_output(run_sobelf_mpi(input_file, output_file, ranks, nodes))

def profile_omp(input_file, output_file, threads=16):
    return parse_sobel_filter_output(run_sobelf_omp(input_file, output_file, threads))

def profile_hyb(input_file, output_file, ranks=4, threads=4, nodes=1):
    return parse_sobel_filter_output(run_sobelf_hyb(input_file, output_file, ranks, threads, nodes))

def profile_cuda(input_file, output_file):
    return parse_sobel_filter_output(run_sobelf_cuda(input_file, output_file))


from functools import lru_cache

@lru_cache(maxsize=None)
def cached_profile_mpi(gif_file, ranks=16, nodes=1):
    return profile_mpi(gif_file, gif_file, ranks, nodes)

@lru_cache(maxsize=None)
def cached_profile_omp(gif_file, thread):
    return profile_omp(gif_file, gif_file, thread)

@lru_cache(maxsize=None)
def cached_profile_hyb(gif_file, rank, thread, node):
    return profile_hyb(gif_file, gif_file, rank, thread, node)

@lru_cache(maxsize=None)
def cached_profile_cuda(gif_file):
    return profile_cuda(gif_file, gif_file)

In [4]:
def get_gif_info(path, input_dir=INPUT_DIRECTORY):
    sizes = []
    with Image.open(os.path.join(input_dir, path)) as im:
        num_frames = im.n_frames
        for frame_index in range(num_frames):
            im.seek(frame_index)
            sizes.append(im.size)
    return num_frames, sizes

In [5]:
example_file = "200_s.gif"

print(profile_omp(example_file, example_file))
print(profile_mpi(example_file, example_file))
print(profile_hyb(example_file, example_file))
print(profile_cuda(example_file, example_file))
print(get_gif_info(example_file))


0.001587
0.308065
0.002756
0.003078
(1, [(200, 200)])


In [6]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier, export_text

import itertools
import random

from tqdm import tqdm

feature_names = ["num_frames","width","pixel","total_pixels","ranks","threads","nodes","cuda_available"]

labels =          {"MPI":0, "OMP":1, "HYB":2, "CUDA":3}
implementations = {0:"MPI", 1:"OMP", 2:"HYB", 3:"CUDA"}

def sample_configs(rank_list, thread_list, node_list, cuda_list, sample_ratio=0.3):

    configs = list(itertools.product(rank_list, thread_list, node_list, cuda_list))

    n_samples = max(1, int(len(configs) * sample_ratio))

    return random.sample(configs, n_samples)

def create_data_set(input_dir, rank_list, thread_list, node_list, cuda_list, sample_ratio=0.3):
    # X : num_frames, width, height, ranks, threads, nodes, cuda_available
    # y : {0 : MPI , 1 : OMP , 2 : HYB , 3 : CUDA}
        
    X = []
    y = []

    errors = []

    gif_files = [f for f in os.listdir(input_dir) if f.lower().endswith(".gif")]

    configs = sample_configs(rank_list, thread_list, node_list, cuda_list, sample_ratio)

    total = len(gif_files) * len(configs)

    with tqdm(total=total) as progression:
        for gif_file in gif_files:
            
            num_frames, sizes = get_gif_info(gif_file, input_dir=input_dir)
            width, height = sizes[0] # assumes all frames have the same size

            for rank, thread, node, cuda in configs:
                times = {}

                # MPI
                try:
                    times["MPI"]  = cached_profile_mpi(gif_file, rank, node)
                except Exception as e:
                    times["MPI"] = np.inf
                    errors.append({
                        "gif": gif_file,
                        "impl": "MPI",
                        "rank": rank,
                        "thread": thread,
                        "node": node,
                        "cuda": cuda,
                        "stderr": str(e)
                    })

                # OMP
                try:
                    times["OMP"]  = cached_profile_omp(gif_file, thread)
                except Exception as e:
                    times["OMP"] = np.inf
                    errors.append({
                        "gif": gif_file,
                        "impl": "OMP",
                        "rank": rank,
                        "thread": thread,
                        "node": node,
                        "cuda": cuda,
                        "stderr": str(e)
                    })

                # HYBRID
                try:
                    times["HYB"]  = cached_profile_hyb(gif_file, rank, thread, node)
                except Exception as e:
                    times["HYB"] = np.inf
                    errors.append({
                        "gif": gif_file,
                        "impl": "HYB",
                        "rank": rank,
                        "thread": thread,
                        "node": node,
                        "cuda": cuda,
                        "stderr": str(e)
                    })

                # CUDA
                try:
                    if cuda:
                        times["CUDA"] = cached_profile_cuda(gif_file)
                    else:
                        times["CUDA"] = np.inf
                except Exception as e:
                    times["CUDA"] = np.inf
                    errors.append({
                        "gif": gif_file,
                        "impl": "CUDA",
                        "rank": rank,
                        "thread": thread,
                        "node": node,
                        "cuda": cuda,
                        "stderr": str(e)
                    })

                best_impl = min(times, key=times.get)

                best_label = labels[best_impl]

                pixel = width * height
                total_pixel = pixel * num_frames

                features = [num_frames, width, pixel, total_pixel, rank, thread, node, cuda]

                X.append(features)
                y.append(best_label)

                progression.update(1)

    return np.array(X), np.array(y), errors

def build_tree(X, y, max_depth):
    tree = DecisionTreeClassifier(max_depth=max_depth)
    tree.fit(X,y)
    return tree

def print_tree(tree):
    tree_text = export_text(tree, feature_names=feature_names)
    print("Decision tree :")
    print(tree_text)
    

ranks = [1,2,4,8]
threads = [2,4,8]
nodes = [1,2]
cudas = [True, False]

max_depth = 5 # TODO : test

X, y, errors = create_data_set(INPUT_DIRECTORY, ranks, threads, nodes, cudas, sample_ratio=0.25)

tree = build_tree(X, y, 5)

print_tree(tree)


100%|██████████| 216/216 [17:05<00:00,  4.75s/it]

Decision tree :
|--- cuda_available <= 0.50
|   |--- pixel <= 92750.00
|   |   |--- ranks <= 6.00
|   |   |   |--- ranks <= 1.50
|   |   |   |   |--- width <= 115.00
|   |   |   |   |   |--- class: 0
|   |   |   |   |--- width >  115.00
|   |   |   |   |   |--- class: 1
|   |   |   |--- ranks >  1.50
|   |   |   |   |--- total_pixels <= 49700.00
|   |   |   |   |   |--- class: 0
|   |   |   |   |--- total_pixels >  49700.00
|   |   |   |   |   |--- class: 0
|   |   |--- ranks >  6.00
|   |   |   |--- num_frames <= 17.00
|   |   |   |   |--- class: 2
|   |   |   |--- num_frames >  17.00
|   |   |   |   |--- class: 1
|   |--- pixel >  92750.00
|   |   |--- total_pixels <= 1057446.00
|   |   |   |--- ranks <= 1.50
|   |   |   |   |--- class: 1
|   |   |   |--- ranks >  1.50
|   |   |   |   |--- ranks <= 3.00
|   |   |   |   |   |--- class: 1
|   |   |   |   |--- ranks >  3.00
|   |   |   |   |   |--- class: 1
|   |   |--- total_pixels >  1057446.00
|   |   |   |--- ranks <= 6.00
|   |   |

In [7]:
print(errors)

[]


In [8]:
for x_, y_ in zip(X, y):
    print(x_, "->", implementations[y_])

[      1    1484 1354892 1354892       2       4       2       0] -> OMP
[      1    1484 1354892 1354892       1       8       2       0] -> OMP
[      1    1484 1354892 1354892       4       8       2       1] -> CUDA
[      1    1484 1354892 1354892       4       4       1       0] -> OMP
[      1    1484 1354892 1354892       2       8       1       1] -> CUDA
[      1    1484 1354892 1354892       1       4       2       1] -> CUDA
[      1    1484 1354892 1354892       8       4       2       1] -> CUDA
[      1    1484 1354892 1354892       2       8       2       0] -> OMP
[      1    1484 1354892 1354892       8       4       2       0] -> OMP
[      1    1484 1354892 1354892       8       4       1       1] -> CUDA
[      1    1484 1354892 1354892       2       4       1       1] -> CUDA
[      1    1484 1354892 1354892       1       4       1       0] -> OMP
[     10     500  250000 2500000       2       4       2       0] -> OMP
[     10     500  250000 2500000       1     